PROJECT 1: ANALYSIS OF HUMAN ACCELERATED REGION 1 (HAR1)

In [ ]:
import sys
import os
import json
import time
import re
import requests
import shutil
from liftover import ChainFile


sys.path.append(os.path.abspath('..'))


from utils import (read_fasta, get_percent_identity, find_mutations, calculate_GC_content, count_bases, count_deaminations, get_all_percent_identity, find_conserved_positions, 
    fetch_dna,analyze_mutations)

print("All packages and custom tools from utils.py loaded successfully!")

data = read_fasta("../data/har1.fasta")
human_chimp_percent_identity = get_percent_identity(data["human"], data["chimp"])

chimp_macaque_percent_identity = get_percent_identity(data["macaque"], data["chimp"])
print(human_chimp_percent_identity)
print(chimp_macaque_percent_identity)


All packages and custom tools from utils.py loaded successfully!
96.89119170984456
100.0


Chimps and humans have ~97% similarities in their HAR, but chimps and macaques are fully similar. Chimps and humans split only 6 million years ago vs chimp/macaque split about 25 million years. Humans' genetic makeup in these regions changed very quickly.

In [2]:

find_mutations(data["human"], data["chimp"])


Position | Seq1 | Seq2
---------------------
  41      |  G   |  A
  66      |  C   |  T
  91      |  G   |  A
  137      |  C   |  T
  153      |  C   |  T
  170      |  C   |  T


[(41, 'G', 'A'),
 (66, 'C', 'T'),
 (91, 'G', 'A'),
 (137, 'C', 'T'),
 (153, 'C', 'T'),
 (170, 'C', 'T')]

C to T transitions can be natural from deamination, but also indicate evolutionary shifts

In [3]:

data = read_fasta("../data/har1.fasta")

human_gc_content = calculate_GC_content(data["human"])
chimp_gc_content = calculate_GC_content(data["chimp"])

print(f"Human HAR1 GC content: {human_gc_content: .2f}%")
print(f"Chimp HAR1 GC content: {chimp_gc_content: .2f}%")

Human HAR1 GC content:  67.88%
Chimp HAR1 GC content:  64.77%


Human HAR1 is more GC rich! 

In [4]:

data = read_fasta("../data/har1.fasta") 
results = get_all_percent_identity(data)
print(results)

human vs chimp: 96.89%
human vs macaque: 96.89%
chimp vs human: 96.89%
chimp vs macaque: 100.00%
macaque vs human: 96.89%
macaque vs chimp: 100.00%
{'human vs chimp': 96.89119170984456, 'human vs macaque': 96.89119170984456, 'chimp vs human': 96.89119170984456, 'chimp vs macaque': 100.0, 'macaque vs human': 96.89119170984456, 'macaque vs chimp': 100.0}


In [5]:
data = read_fasta("../data/har1.fasta") 

conservation = find_conserved_positions(data)
print(f"Region conservation: {conservation:.2f}%")

Region conservation: 96.89%


MULTIPLE SEQUENCE ANALYSIS OF 312 HARs in HUMANS VS CHIMPS

In [6]:
import requests
import gzip
import shutil
import os


if not os.path.exists('chains'):
    os.makedirs('chains')


url = "https://hgdownload.soe.ucsc.edu/goldenPath/hg38/liftOver/hg38ToPanTro6.over.chain.gz"
local_gz = "chains/hg38ToPanTro6.over.chain.gz"

print("Downloading the Human-to-Chimp mapping file...")
r = requests.get(url, stream=True)
with open(local_gz, 'wb') as f:
    shutil.copyfileobj(r.raw, f)

print("Mapping file downloaded successfully.")

Mapping file downloaded successfully.


In [4]:

chain_path = os.path.abspath("../data/chains/hg38ToPanTro6.over.chain.gz")
converter = ChainFile(chain_path, one_based=False)


h_chrom = "chr2"
h_pos = 235865386

print(f"Human Address: {h_chrom}:{h_pos}")


chimp_results = converter[h_chrom][h_pos]

if chimp_results:
    c_chrom, c_pos, c_strand = chimp_results[0]
    print(f"Successfully lifted to Chimp!")
    print(f"Chimp Address: {c_chrom}:{c_pos}")
else:
    print("Could not find this region in the Chimp genome (it might be a human-specific insertion).")

Human Address: chr2:235865386
Successfully lifted to Chimp!
Chimp Address: chr2B:122441811


CHROMOSOME 2 IN HUMANS IS A FUSION OF 2 ANCESTRAL CHROMOSOMES, by showing that chr2 in humans is chr2B in chimps, we see that our lifting/converting worked!!!

In [5]:
all_hars = []

with open("../data/zooHARs_hg38.bed", "r") as f:
    for line in f:
        parts = line.strip().split('\t')
        if len(parts) < 4:
            continue
        chrom = parts[0]
        start = parts[1]
        end = parts[2]
        name = parts[3]
        all_hars.append({"name": name, "chrom": chrom, "start": start, "end": end})

print(f"Successfully loaded {len(all_hars)} HARs.")
print(f"First one: {all_hars[0]}")

Successfully loaded 312 HARs.
First one: {'name': 'ZOOHAR.1', 'chrom': 'chr2', 'start': '235865386', 'end': '235865436'}


In [6]:
converter = ChainFile(chain_path, one_based=False)

def fetch_dna(genome, chrom, pos_start, pos_end):
    url = f"https://genome.ucsc.edu/cgi-bin/das/{genome}/dna?segment={chrom}:{pos_start},{pos_end}"
    try:
        r = requests.get(url, timeout=10)
        match = re.search(r'<DNA.*?>(.*?)</DNA>', r.text, re.DOTALL)
        if match:
            return match.group(1).replace('\n', '').replace(' ', '').upper()
    except:
        return None
    return None


final_comparisons = []

print("Starting the Smart Fetcher...")

for har in all_hars[:5]:
    h_chrom = har['chrom']
    h_start = int(har['start'])
    h_end = int(har['end'])
    
    lift_start = converter[h_chrom][h_start]
    lift_end = converter[h_chrom][h_end]
    
    if lift_start and lift_end:
        c_chrom = lift_start[0][0]
        c_start = lift_start[0][1]
        c_end = lift_end[0][1]
        
        print(f"Lifting {har['name']}: Human {h_chrom} -> Chimp {c_chrom}")
        
        
        h_seq = fetch_dna("hg38", h_chrom, h_start, h_end)
        c_seq = fetch_dna("panTro6", c_chrom, c_start, c_end)
        
        if h_seq and c_seq:
            final_comparisons.append({"name": har['name'],"human_seq": h_seq,"chimp_seq": c_seq})
    
    time.sleep(0.5)


print("\n--- FINAL VERIFICATION ---")
demo = final_comparisons[0]
print(f"Region: {demo['name']}")
print(f"Human: {demo['human_seq'][:60]}")
print(f"Chimp: {demo['chimp_seq'][:60]}")



Starting the Smart Fetcher...
Lifting ZOOHAR.1: Human chr2 -> Chimp chr2B
Lifting ZOOHAR.2: Human chr13 -> Chimp chr13
Lifting ZOOHAR.3: Human chr5 -> Chimp chr5
Lifting ZOOHAR.4: Human chr6 -> Chimp chr6
Lifting ZOOHAR.5: Human chr2 -> Chimp chr2A

--- FINAL VERIFICATION ---
Region: ZOOHAR.1
Human: CCCACAGTAACACGTGTGGCGCCGACCCCGCCGTGCGCAATCGGGGCTTTA
Chimp: TCCACAATAACAAGTGTGTCACTAACCCCGCCGTGCATAATCGGGGCTTTA


In [7]:
lifted_hars = []

for har in all_hars:
    h_chrom = har["chrom"]
    h_start = int(har["start"])
    h_end = int(har["end"])

    start_result = converter[h_chrom][h_start]
    end_result = converter[h_chrom][h_end]

    if start_result and end_result:
        lifted_hars.append({
            "name": har["name"],
            "human_chrom": h_chrom,
            "human_start": h_start,
            "human_end": h_end,
            "chimp_chrom": start_result[0][0],
            "chimp_start": start_result[0][1],
            "chimp_end": end_result[0][1],
            "strand": start_result[0][2]
        })

print(f"Successfully lifted {len(lifted_hars)} of {len(all_hars)} HARs.")
print(lifted_hars[:3])

Successfully lifted 306 of 312 HARs.
[{'name': 'ZOOHAR.1', 'human_chrom': 'chr2', 'human_start': 235865386, 'human_end': 235865436, 'chimp_chrom': 'chr2B', 'chimp_start': 122441811, 'chimp_end': 122441861, 'strand': '+'}, {'name': 'ZOOHAR.2', 'human_chrom': 'chr13', 'human_start': 72540830, 'human_end': 72541395, 'chimp_chrom': 'chr13', 'chimp_start': 53783582, 'chimp_end': 53784148, 'strand': '+'}, {'name': 'ZOOHAR.3', 'human_chrom': 'chr5', 'human_start': 77595138, 'human_end': 77595554, 'chimp_chrom': 'chr5', 'chimp_start': 37244366, 'chimp_end': 37243949, 'strand': '-'}]


In [8]:
sequences = []
failed = []

print(f"Fetching DNA for {len(lifted_hars)} HARs. This will take ~10 minutes...")

for i, har in enumerate(lifted_hars):
    
    h_seq = fetch_dna("hg38", har["human_chrom"], har["human_start"], har["human_end"])
    c_seq = fetch_dna("panTro6", har["chimp_chrom"], har["chimp_start"], har["chimp_end"])
    
    if h_seq and c_seq:
        sequences.append({
            "name": har["name"],
            "human_seq": h_seq,
            "chimp_seq": c_seq
        })
    else:
        failed.append(har["name"])
    
    
    if (i + 1) % 50 == 0:
        print(f"  {i + 1}/{len(lifted_hars)} done...")
    
    time.sleep(0.4)

print(f"\nDone! Successfully fetched {len(sequences)} sequence pairs.")
print(f"Failed: {len(failed)}")


import json
with open("../data/har_sequences.json", "w") as f:
    json.dump(sequences, f, indent=4)
print(f"CRITICAL SYNC: {len(sequences)} sequences saved to disk.")

Fetching DNA for 306 HARs. This will take ~10 minutes...
  50/306 done...
  100/306 done...
  150/306 done...
  200/306 done...
  250/306 done...
  300/306 done...

Done! Successfully fetched 281 sequence pairs.
Failed: 25
CRITICAL SYNC: 281 sequences saved to disk.


In [ ]:
data_path = '../data/har_sequences.json'
output_path = '../data/har_mutation_results.json'

with open(data_path, 'r') as f:
    data = json.load(f)

final_results = []

for entry in data:
    mutation_stats = analyze_mutations(entry['human_seq'], entry['chimp_seq'])
    final_results.append({**entry, **mutation_stats})

with open(output_path, 'w') as f:
    json.dump(final_results, f, indent=4)

print(f"Success! Processed {len(final_results)} regions.")
print(f"Results saved to: {output_path}")

Success! Processed 281 regions.
Results saved to: ../data/har_mutation_results.json


In [ ]:
top_bias = sorted(final_results, key=lambda x: x['bias_ratio'], reverse=True)[0]

print(f"Top gBGC Candidate: {top_bias['name']}")
print(f"Bias Ratio: {top_bias['bias_ratio']:.2f}")
print(f"Total Mutations: {top_bias['subs']}")

Top gBGC Candidate: ZOOHAR.1
Bias Ratio: 10.00
Total Mutations: 9


ZOOHAR.1 had the most GC-biased gene conversions, showing that these weak to strong mutations likely arose due to non adaptive DNA-repair mechanism glitches and not selective mutation due to natural selection/positive selection. 